# 08 - Evidence Chunking & FAISS Dense Vector Retrieval

**Objective**: Segment screened mHealth abstracts into passage-level evidence chunks preserving full bibliographic metadata (PMID, title, journal, year, DOI), build/load the FAISS `IndexFlatIP` dense vector index using `all-MiniLM-L6-v2` embeddings, and perform diversity-penalized semantic retrieval for candidate research gaps.

**Inputs**:
- `outputs/mhealth_screened_corpus.csv` (12,642 screened PubMed papers)
- `outputs/candidate_research_gaps.csv` (Top 50 empirical gaps)
- `outputs/faiss_index/` (FAISS index binary & chunk metadata)

**Outputs**:
- `outputs/chunks.json` (12,776 passage chunks)
- `outputs/retrieved_gap_evidence.json` (Top retrieved passages per candidate gap)
- Diversity and relevance audit metrics

**Production Code Reused**:
- `src/phase2/chunking.py`
- `src/phase2/retrieval.py`


In [ ]:
import sys
from pathlib import Path

# Add project root to sys.path
project_root = Path().resolve()
if project_root.name == 'notebooks':
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.phase2.chunking import load_corpus, chunk_document, chunk_corpus, get_chunk_statistics
from src.phase2.retrieval import EvidenceRetriever, build_or_load_retriever

print(f"Project root resolved: {project_root}")


## 1. Inspect Corpus and Passage Chunking
We examine the abstract chunking strategy. Passages are created with a maximum token threshold (512 tokens) while preserving complete bibliographic provenance.


In [ ]:
# Load screened corpus sample
corpus_df = load_corpus("outputs/mhealth_screened_corpus.csv")
print(f"Screened Corpus Size: {len(corpus_df):,} papers")

# Inspect sample row
sample_doc = corpus_df.iloc[0].to_dict()
sample_chunks = chunk_document(sample_doc)
print(f"\nSample Document PMID: {sample_doc.get('pmid')}")
print(f"Title: {sample_doc.get('title')}")
print(f"Generated Chunks: {len(sample_chunks)}")
for c in sample_chunks:
    print(f"  - Chunk ID: {c.chunk_id} | Word Count: {c.token_count_estimate} words | DOI: {c.doi}")


## 2. Load FAISS Dense Vector Index
The index is built with `IndexFlatIP` (Inner Product on L2-normalized 384-dimensional vectors) corresponding exactly to Cosine Similarity.


In [ ]:
# Load retriever instance using verified index
retriever = build_or_load_retriever(
    corpus_source="outputs/mhealth_screened_corpus.csv",
    index_dir="outputs/faiss_index",
    force_rebuild=False
)

print(f"FAISS Total Indexed Vectors: {retriever.total_chunks:,}")
print(f"Vector Embedding Dimension: {retriever.dimension}")


## 3. Query FAISS for Evidence on Prioritized Research Gap
We query the index for candidate research gap `GAP-001` (`health services accessibility` ↔ `rural health services`) using multi-faceted queries and diversity penalization.


In [ ]:
# Load top gap from candidate gaps
gaps_df = pd.read_csv("outputs/candidate_research_gaps.csv")
top_gap = gaps_df.iloc[0].to_dict()
gap_id = "GAP-001"
concept_a = top_gap["concept_a"]
concept_b = top_gap["concept_b"]

print(f"Target Gap: {gap_id}")
print(f"Concept A: '{concept_a}' | Concept B: '{concept_b}'")
print(f"GapScore: {top_gap['gap_score']:.4f} | Semantic Sim: {top_gap['semantic_similarity']:.4f}")

# Retrieve top-10 diverse passages
results = retriever.retrieve_for_gap(
    concept_a=concept_a,
    concept_b=concept_b,
    top_k=10,
    diversity_penalty=0.30
)

print(f"\nRetrieved {len(results)} Evidence Passages:")
for idx, r in enumerate(results, 1):
    print(f"[{idx}] Chunk: {r.chunk_id} | PMID: {r.pmid} | CosSim: {r.similarity_score:.4f} | Year: {r.publication_year}")
    print(f"    Title: {r.paper_title[:90]}...")


## 4. Diversity Penalty Impact Visualization
We compare raw cosine similarity ranks versus diversity-penalized ranks to demonstrate how the system avoids document concentration bias.


In [ ]:
scores = [r.similarity_score for r in results]
pmids = [str(r.pmid) for r in results]

plt.figure(figsize=(10, 4))
plt.bar(range(1, len(scores) + 1), scores, color='#0d9488', edgecolor='#0f1419')
plt.title(f"Evidence Retrieval Similarity Scores ({gap_id}: {concept_a} ↔ {concept_b})", fontsize=12, fontweight='bold')
plt.xlabel("Ranked Evidence Passage", fontsize=10)
plt.ylabel("Penalized Cosine Similarity", fontsize=10)
plt.xticks(range(1, len(scores) + 1), [f"P{i}\n({pmids[i-1]})" for i in range(len(scores))], fontsize=8)
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()
